# Decoding Strategies for Language Models — Kubeflow Edition

This notebook demonstrates five decoding strategies using a small language model.

**Strategies covered:**
1. **Greedy** — always pick the highest-probability token
2. **Beam Search** — maintain top-k candidate sequences simultaneously
3. **Top-k Sampling** — sample from the top-k highest-probability tokens
4. **Top-p (Nucleus) Sampling** — sample from the smallest token set covering probability mass p
5. **Speculative Decoding** — draft tokens with a small model, verify with the large model

**Prerequisites (one-time setup per pod)**

```bash
bash ~/setup_env.sh   # installs pinned course baseline
# then: Kernel -> Restart Kernel
python ~/check_env.py  # should show all checks passing
```

## 1. Environment check

In [ ]:
import sys, importlib, torch

REQUIRED = ['torch', 'transformers']
missing = [p for p in REQUIRED if not importlib.util.find_spec(p)]
if missing:
    raise RuntimeError(f'Missing: {missing}. Run bash ~/setup_env.sh in a terminal, then Restart Kernel.')

print(f'Python       : {sys.version.split()[0]}')
print(f'Torch        : {torch.__version__}')
print(f'CUDA visible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU. Decoding works on CPU but is slower. Speculative decoding requires GPU.')

## 2. Load model and tokenizer

We use **TinyLlama-1.1B** as our main model. It is small enough to fit on any GPU or even CPU for demo purposes.
For speculative decoding we also load a tiny 'draft' model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MAIN_MODEL_ID  = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
DRAFT_MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # same model as draft for demo
# In production you would use a much smaller draft model, e.g. a 125M distilled variant.

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(MAIN_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading main model ...')
model = AutoModelForCausalLM.from_pretrained(
    MAIN_MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None,
)
model.eval()
print(f'Model loaded on {DEVICE}')

## 3. Define the prompt

A short open-ended prompt works well for showing differences between strategies.

In [ ]:
PROMPT = (
    "<|system|>\nYou are a helpful assistant.</s>\n"
    "<|user|>\nExplain the water cycle in simple terms.</s>\n"
    "<|assistant|>\n"
)

inputs = tokenizer(PROMPT, return_tensors='pt').to(DEVICE)
input_len = inputs['input_ids'].shape[1]
print(f'Prompt token length: {input_len}')

## 4. Greedy Decoding

**Algorithm:** At each step select the token with the **highest probability**.

```
next_token = argmax P(token | context)
```

**Properties:**
- Deterministic (same output every run)
- Fast (no branching)
- Can get stuck in repetitive loops
- Best for: factual Q&A, structured output where diversity is unwanted

In [ ]:
import time

@torch.inference_mode()
def decode_greedy(model, inputs, max_new_tokens=80):
    t0 = time.time()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,          # greedy
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - t0
    new_tokens = out[0, input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    tps  = len(new_tokens) / elapsed
    print(f'[Greedy]  {len(new_tokens)} tokens  |  {elapsed:.2f}s  |  {tps:.1f} tok/s')
    print(text)
    return text

greedy_out = decode_greedy(model, inputs)

## 5. Beam Search

**Algorithm:** Keep the **top-B candidate sequences** (beams) at each step and expand them all.

```
At each step:
  For each beam b, compute P(token | b)
  Keep top-B (beam, token) pairs by cumulative log-prob
```

**Properties:**
- Deterministic
- Finds a better overall sequence than greedy (globally more probable)
- More compute: B × forward passes worth of work
- Can produce generic/bland output ('safe' generations)
- Best for: machine translation, structured summarisation, code generation

**Key parameters:** `num_beams`, `length_penalty`, `no_repeat_ngram_size`

In [ ]:
@torch.inference_mode()
def decode_beam(model, inputs, max_new_tokens=80, num_beams=4,
                length_penalty=1.0, no_repeat_ngram_size=3):
    t0 = time.time()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=num_beams,
        length_penalty=length_penalty,
        no_repeat_ngram_size=no_repeat_ngram_size,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - t0
    new_tokens = out[0, input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    tps  = len(new_tokens) / elapsed
    print(f'[Beam]  beams={num_beams}  {len(new_tokens)} tokens  |  {elapsed:.2f}s  |  {tps:.1f} tok/s')
    print(text)
    return text

beam_out = decode_beam(model, inputs)

## 6. Top-k Sampling

**Algorithm:** At each step, restrict the vocabulary to the **k** highest-probability tokens, re-normalise, and sample.

```
candidates = top_k_tokens(vocab, k)
p_renorm   = softmax(logits[candidates] / temperature)
next_token = sample(candidates, p_renorm)
```

**Properties:**
- Stochastic — different output each run
- `temperature > 1` flattens the distribution (more random)
- `temperature < 1` sharpens it (more greedy-like)
- Larger k → more diversity; k=1 → greedy
- Weakness: k is fixed regardless of how peaked/flat the distribution actually is
- Best for: creative text, story generation, chatbots

**Key parameters:** `top_k`, `temperature`

In [ ]:
@torch.inference_mode()
def decode_topk(model, inputs, max_new_tokens=80, top_k=50, temperature=0.8):
    t0 = time.time()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=top_k,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - t0
    new_tokens = out[0, input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    tps  = len(new_tokens) / elapsed
    print(f'[Top-k]  k={top_k}  temp={temperature}  {len(new_tokens)} tokens  |  {elapsed:.2f}s  |  {tps:.1f} tok/s')
    print(text)
    return text

topk_out = decode_topk(model, inputs)

## 7. Top-p (Nucleus) Sampling

**Algorithm:** At each step, take the **smallest set of tokens whose cumulative probability >= p**, re-normalise, and sample.

```
sort tokens by descending probability
take tokens until cumsum >= p
p_renorm   = softmax(logits[nucleus] / temperature)
next_token = sample(nucleus, p_renorm)
```

**Properties:**
- Adaptive: the nucleus size changes each step based on the actual distribution shape
- When the model is confident, the nucleus is small (focused)
- When the model is uncertain, the nucleus is large (diverse)
- Often combined with top_k for a dual filter
- Best for: open-ended dialogue, creative writing

**Key parameters:** `top_p`, `temperature`, (optionally `top_k` as a cap)

In [ ]:
@torch.inference_mode()
def decode_topp(model, inputs, max_new_tokens=80, top_p=0.9, temperature=0.8, top_k=0):
    t0 = time.time()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=top_p,
        top_k=top_k,     # 0 disables the k-filter; set >0 to combine with top-p
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - t0
    new_tokens = out[0, input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    tps  = len(new_tokens) / elapsed
    print(f'[Top-p]  p={top_p}  temp={temperature}  {len(new_tokens)} tokens  |  {elapsed:.2f}s  |  {tps:.1f} tok/s')
    print(text)
    return text

topp_out = decode_topp(model, inputs)

## 8. Speculative Decoding

**Algorithm:**
1. A small **draft model** generates K candidate tokens quickly.
2. The large **target model** evaluates all K tokens in one forward pass.
3. Tokens are **accepted** if the target model agrees; mismatches are corrected.

```
draft_tokens = draft_model.generate(context, k=5)   # fast
logits_target = target_model(context + draft_tokens) # one forward pass
accepted = [t for t in draft_tokens if accept_criterion(t)]
```

**Properties:**
- Produces **identical distribution** to the target model (lossless)
- Speedup proportional to draft-model acceptance rate
- Requires two models loaded simultaneously (more memory)
- Best for: production inference where throughput matters

**Note:** For this demo we simulate speculative decoding using the same model as both draft and target to illustrate the mechanics without needing two different models.

In [ ]:
# ── Speculative decoding simulation ─────────────────────────────────────────
# In a real deployment: draft_model is a ~100M model, target_model is 7B+.
# Here we use the same model for both to keep VRAM low and illustrate the API.

import torch.nn.functional as F

@torch.inference_mode()
def speculative_decode(
    draft_model, target_model, tokenizer, inputs,
    max_new_tokens=80, K=4, temperature=1.0
):
    """
    Simplified speculative decoding.
    K: number of draft tokens to generate per round.
    """
    device = inputs['input_ids'].device
    input_ids = inputs['input_ids'].clone()
    accepted_total = 0
    rounds = 0
    t0 = time.time()

    while (input_ids.shape[1] - inputs['input_ids'].shape[1]) < max_new_tokens:
        rounds += 1
        context_len = input_ids.shape[1]

        # 1. Draft model generates K tokens
        draft_out = draft_model.generate(
            input_ids,
            max_new_tokens=K,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-6),
            pad_token_id=tokenizer.eos_token_id,
        )
        draft_tokens = draft_out[0, context_len:]  # shape (K,)
        if len(draft_tokens) == 0:
            break

        # 2. Target model scores the draft continuation in one forward pass
        candidate_ids = torch.cat([input_ids, draft_tokens.unsqueeze(0)], dim=1)
        target_logits = target_model(candidate_ids).logits  # (1, context+K, vocab)

        # 3. Accept / reject each draft token
        n_accepted = 0
        for i, draft_tok in enumerate(draft_tokens):
            pos_logits = target_logits[0, context_len + i - 1, :]
            target_prob = F.softmax(pos_logits / max(temperature, 1e-6), dim=-1)
            draft_prob  = target_prob[draft_tok].item()   # simplified acceptance
            # Acceptance criterion: accept with probability min(1, target_p / draft_p)
            # (here draft_p approximated as target_p for the same model)
            if draft_prob > 0.05:   # simplified threshold for demo
                n_accepted += 1
                input_ids = torch.cat([input_ids, draft_tok.view(1, 1)], dim=1)
            else:
                # Reject: sample a correction token from the target
                corr = torch.multinomial(target_prob, 1)
                input_ids = torch.cat([input_ids, corr.view(1, 1)], dim=1)
                break

        accepted_total += n_accepted

        if input_ids[0, -1].item() == tokenizer.eos_token_id:
            break

    elapsed = time.time() - t0
    new_tokens = input_ids[0, inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    tps  = len(new_tokens) / elapsed
    accept_rate = accepted_total / max(rounds * K, 1)
    print(f'[Speculative]  K={K}  rounds={rounds}  accept_rate={accept_rate:.2f}')
    print(f'               {len(new_tokens)} tokens  |  {elapsed:.2f}s  |  {tps:.1f} tok/s')
    print(text)
    return text

if DEVICE == 'cuda':
    spec_out = speculative_decode(model, model, tokenizer, inputs, max_new_tokens=80, K=4)
else:
    print('Speculative decoding skipped on CPU (requires GPU for meaningful speedup).')
    spec_out = None

## 9. Side-by-side comparison

In [ ]:
print('=' * 70)
print(' DECODING STRATEGY COMPARISON')
print('=' * 70)

outputs_map = {
    "Greedy"    : greedy_out,
    "Beam (4)"  : beam_out,
    "Top-k (50)": topk_out,
    "Top-p (0.9)": topp_out,
}
if spec_out:
    outputs_map['Speculative'] = spec_out

for name, text in outputs_map.items():
    print(f'\n--- {name} ---')
    print(text[:300] + ('...' if len(text) > 300 else ''))
print('\n' + '=' * 70)

## 10. Visualise token probability distributions

This cell shows the raw probability mass for the **first generation step** under each setting. It helps you see concretely what 'greedy', 'top-k' and 'nucleus' mean on actual logits.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

@torch.inference_mode()
def get_first_step_probs(model, inputs, temperature=1.0):
    logits = model(**inputs).logits[0, -1, :]
    probs  = torch.softmax(logits / temperature, dim=-1)
    return probs.cpu().numpy()

probs = get_first_step_probs(model, inputs)
sorted_idx  = np.argsort(probs)[::-1]
top50_probs = probs[sorted_idx[:50]]
top50_tokens = [tokenizer.decode([i]) for i in sorted_idx[:50]]

# --- Greedy: argmax ---
# --- Top-k=10 cutoff ---
k = 10
cumsum = np.cumsum(top50_probs)
p_cutoff = 0.9   # for top-p illustration
nucleus_end = np.searchsorted(cumsum, p_cutoff) + 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Greedy
axes[0].bar(range(20), top50_probs[:20], color='steelblue')
axes[0].bar(0, top50_probs[0], color='crimson', label=f'Greedy pick: {top50_tokens[0]!r}')
axes[0].set_title('Greedy — pick argmax')
axes[0].set_xlabel('Token rank')
axes[0].set_ylabel('Probability')
axes[0].legend(fontsize=8)

# Top-k
colors_k = ['orange' if i < k else 'lightgray' for i in range(20)]
axes[1].bar(range(20), top50_probs[:20], color=colors_k)
axes[1].set_title(f'Top-k (k={k}) — orange = candidate')
axes[1].set_xlabel('Token rank')

# Top-p nucleus
colors_p = ['mediumseagreen' if i < nucleus_end else 'lightgray' for i in range(min(30, len(top50_probs)))]
axes[2].bar(range(len(colors_p)), top50_probs[:len(colors_p)], color=colors_p)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title(f'Top-p (p={p_cutoff}) — green = nucleus ({nucleus_end} tokens)')
axes[2].set_xlabel('Token rank')

plt.tight_layout()
plt.savefig('decoding_distributions.png', dpi=120)
plt.show()
print('Figure saved to decoding_distributions.png')

## 11. Strategy summary table

| Strategy | Deterministic | Diversity | Speed | Best for |
|----------|:---:|:---:|:---:|---|
| Greedy | Yes | None | Fastest | Factual Q&A, structured output |
| Beam Search | Yes | Low | Slow (×B) | Translation, summarisation |
| Top-k Sampling | No | Medium | Fast | Creative text, chatbots |
| Top-p Sampling | No | Adaptive | Fast | Open-ended dialogue |
| Speculative | No* | Same as target | Fastest† | Production inference |

*Speculative decoding is stochastic when temperature>0 but matches the target model's distribution exactly.  
†Speedup depends on draft acceptance rate.

## 12. Practical Lab — Student Exercises

Work through the cells below. Change **only the parameters shown** and rerun to observe the effect.

---

### Exercise 1: Greedy vs temperature

Set `temperature=0.001` (nearly greedy) vs `temperature=2.0` (very random) in top-k.
What happens to coherence and repetition?

In [ ]:
# Lab Exercise 1 — Temperature effect
# Try: temperature = 0.1, 0.5, 1.0, 1.5, 2.0
TEMPERATURE = 1.0   # <-- CHANGE ME

@torch.inference_mode()
def decode_with_temperature(model, inputs, max_new_tokens=80, temperature=1.0):
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True if temperature > 0.01 else False,
        temperature=temperature,
        top_k=0, top_p=1.0,   # no filtering — pure temperature
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0, input_len:], skip_special_tokens=True)
    print(f'[Temperature={temperature}]')
    print(text)

decode_with_temperature(model, inputs, temperature=TEMPERATURE)

### Exercise 2: Beam width effect

Run beam search with `num_beams = 1, 2, 4, 8`. What happens to output quality and inference time?
Is the improvement proportional to the cost?

In [ ]:
# Lab Exercise 2 — Beam width
NUM_BEAMS = 4   # <-- CHANGE ME: try 1, 2, 4, 8

decode_beam(model, inputs, num_beams=NUM_BEAMS)

### Exercise 3: Top-k cutoff

Try `top_k = 1` (greedy), `10`, `50`, `200`, `0` (disabled). When k is very large, how does the output change?

In [ ]:
# Lab Exercise 3 — Top-k cutoff
TOP_K = 50        # <-- CHANGE ME
TEMP  = 0.8

decode_topk(model, inputs, top_k=TOP_K, temperature=TEMP)

### Exercise 4: Top-p nucleus size

Try `top_p = 0.5, 0.7, 0.9, 0.95, 1.0`. What does p=0.5 mean intuitively? How does the nucleus size change between confident and uncertain steps?

In [ ]:
# Lab Exercise 4 — Top-p
TOP_P = 0.9   # <-- CHANGE ME
TEMP  = 0.8

decode_topp(model, inputs, top_p=TOP_P, temperature=TEMP)

### Exercise 5: Combining top-k + top-p

Many production systems use **both** filters together. Try different combos and pick one you like.

Suggested combos to explore:
- k=50, p=0.9 (popular default)
- k=0, p=0.85 (nucleus only)
- k=20, p=1.0 (top-k only)

In [ ]:
# Lab Exercise 5 — Combined top-k + top-p
EX5_TOP_K = 50    # <-- CHANGE ME
EX5_TOP_P = 0.9   # <-- CHANGE ME
EX5_TEMP  = 0.8

decode_topp(model, inputs, top_p=EX5_TOP_P, temperature=EX5_TEMP, top_k=EX5_TOP_K)

### Exercise 6: Repetition penalty

`repetition_penalty > 1.0` penalises tokens that have already appeared. Try values `1.0, 1.1, 1.3, 1.5` on greedy — does it cure repetition? Any downsides?

In [ ]:
# Lab Exercise 6 — Repetition penalty
REP_PENALTY = 1.1   # <-- CHANGE ME: try 1.0, 1.1, 1.3, 1.5

@torch.inference_mode()
def decode_with_rep_penalty(model, inputs, max_new_tokens=80, rep_penalty=1.0):
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=rep_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0, input_len:], skip_special_tokens=True)
    print(f'[Greedy + repetition_penalty={rep_penalty}]')
    print(text)

decode_with_rep_penalty(model, inputs, rep_penalty=REP_PENALTY)

### Exercise 7: Speculative decoding — draft window K

Change the number of draft tokens `K`. How does K affect throughput and acceptance rate?
At what K does the acceptance rate drop noticeably?

In [ ]:
# Lab Exercise 7 — Speculative decoding (GPU only)
SPEC_K = 4    # <-- CHANGE ME: try 2, 4, 8, 16

if DEVICE == 'cuda':
    speculative_decode(model, model, tokenizer, inputs, max_new_tokens=80, K=SPEC_K)
else:
    print('GPU required for speculative decoding demo.')

### Exercise 8: Beam search length penalty

`length_penalty > 1.0` rewards longer sequences; `< 1.0` rewards shorter ones. Try `0.5, 1.0, 1.5, 2.0` and observe how the generated length changes.

In [ ]:
# Lab Exercise 8 — Length penalty in beam search
LEN_PENALTY = 1.0   # <-- CHANGE ME

decode_beam(model, inputs, num_beams=4, length_penalty=LEN_PENALTY)

## 13. Takeaways

- **Greedy** is fast and deterministic but prone to repetitive, locally-optimal traps.
- **Beam search** explores multiple paths — better global quality, but slower and can be too 'safe'.
- **Top-k sampling** introduces diversity by restricting the vocabulary; tune `k` and `temperature` together.
- **Top-p sampling** is adaptive — the nucleus automatically shrinks when the model is confident.
- **Speculative decoding** is a lossless speed trick: identical outputs to the target model, higher throughput.
- In practice, production chatbots typically use `top_k=50, top_p=0.9, temperature≈0.7`.

**Kubeflow notes**
- Figures saved in the current directory are accessible from the JupyterLab file browser.
- For persistent storage across pod restarts, save outputs under `~/`.